[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GaetanKlopocki/French_LLM_Decoder/blob/main/French_LLM_Decoder_training.ipynb)
# Conversational LLM *from scratch* — Decoder-only architecture

This notebook builds, by hand in PyTorch, a conversational language model that follows some of the architectural choices of modern LLMs, and then trains it on a 85M token French chats dataset (https://github.com/GaetanKlopocki/French_LLM_decoder , forked from https://github.com/angeluriot/French_instruct).

**Note for GitHub Users:** This notebook is specifically designed for **training your own model**. A pre-trained version of this model is available on the GitHub repository, along with a separate notebook dedicated to inference for those who wish to use the model without training it themselves.


**Modern components implemented here:**
- **Decoder-only, causal self-attention** — no cross-attention.
- **RoPE (Rotary Position Embeddings)** instead of sinusoidal/learned positional encoding.
- **RMSNorm** instead of LayerNorm.
- **SwiGLU** in the feed-forward instead of a classic GELU MLP.
- **KV Cache** for efficient token-by-token generation.
- Shared embedding weights between input and output head.

**Notebook Structure:**
1. GPU Check + installation
2. Configuration (this is where you can change the parameters, depending on your time and resources)
3. Google Drive (persistent checkpoints)
4. Dataset loading (cloning the GitHub repo)
5. Pair construction (prompt, response)
6. BPE Tokenizer trained from scratch
7. PyTorch Dataset — concatenated sequences + prompt masking in the loss
8. Decoder Transformer model (RoPE, RMSNorm, SwiGLU, KV cache)
9. Initialization
10. Training (with automatic resume)
11. Generation / Interactive Chat (with KV cache)

**⚠️ Realistic Expectations:** An LLM built "from scratch" and trained on a single Colab GPU (T4 free, ~16 GB) remains a house-made project—a few tens of millions of parameters, far from commercial LLMs.

## 1. GPU Check

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("PyTorch :", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

if device.type != "cuda":
    print("⚠️ No GPU detected. If the Notebook is run in Google Colab,  go to Change runtime type and choose T4 GPU (not necessary for inference only)")

## 2. Configuration

In [ ]:
CONFIG = {
    # --- Dataset ---
    "subset_size": None,         # number of chats used (None = entire dataset, ~275k chats)
    "max_history_turns": 6,      # number of messages that can appear as history in the prompt
    "prompt_max_len": 192,       # max length of the prompt, in tokens (context + history)
    "response_max_len": 128,     # max length of the answer, in tokens
    "val_ratio": 0.02,           # ratio (or percent) of the dataset that will be used for validation (0.02 = 2%)

    # --- Tokenizer (BPE trained from scratch) ---
    "vocab_size": 16000,

    # --- Model ---
    "d_model": 384,
    "n_heads": 6,                 # d_model / n_heads must be an even integer (RoPE constraint) -> 384/6=64 OK
    "n_layers": 6,
    "d_ff": 1024,                 # ~2.67 * d_model
    "dropout": 0.1,
    "rope_theta": 10000.0,
    "pos_max_len": 1024,          # RoPE capacity higher than prompt_max_len + response_max_len as it has to encode the generation too

    # --- Training ---
    "batch_size": 32,
    "grad_accum_steps": 2,        # actual batch = batch_size * grad_accum_steps
    "lr": 3e-4,
    "max_steps": 40000,
    "warmup_steps": 2000,         # number of steps with a smaller learnig rate at the beginning, to warm up the model
    "finetune_steps": 10000,      # number of steps with a steeper decline of learning rate at the end, to fine-tune the model
    "label_smoothing": 0.0,       # 0 here as the cross_entropy has ignore_index=-100
    "log_every": 50,
    "save_every": 500,

    # --- Checkpoints ---
    "use_drive": True,            # to track the progress of the training
    "checkpoint_dir": "/content/drive/MyDrive/french_llm_decoder",
}

## 3. Google Drive (to keep the checkpoints during training)

Since a free Colab session can time out at any time, we save the tokenizer and checkpoints to Drive: if the session times out, simply rerun the entire notebook—the training will resume automatically (see Section 10).

In [ ]:
import os

if CONFIG["use_drive"]:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    CONFIG["checkpoint_dir"] = "/content/checkpoints"

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
print("Dossier de checkpoints :", CONFIG["checkpoint_dir"])

## 4. Dataset loading

Architecture of the dataset:

```json
{
  "context": "...",
  "conversation": [{"role": "user", "text": "..."}, {"role": "assistant", "text": "..."}, ...],
  "author": "human" | "chatbot",
  "style": "human" | "chatbot",
  "code": true | false,
  "source": "..."
}
```

In [ ]:
import os

REPO_DIR = "/content/French_LLM_Decoder"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/GaetanKlopocki/French_LLM_Decoder.git {REPO_DIR}
else:
    print("Repository already cloned.")

In [ ]:
import sys, random

sys.path.insert(0, REPO_DIR)
import load as fi_load  # load.py from repository

_cwd = os.getcwd()
os.chdir(REPO_DIR)
raw_data = fi_load.load_dataset()
os.chdir(_cwd)

print("Total number of chats in dataset:", len(raw_data))

random.seed(42)
random.shuffle(raw_data)

if CONFIG["subset_size"] is not None:
    raw_data = raw_data[: CONFIG["subset_size"]]

print("Number of chats used for training:", len(raw_data))
print("\nChat example:")
print(raw_data[0])

## 5. Pairs building (prompt, response)

For each response from the assistant:
- **prompt** = any context + history of previous turns (limited to the last `max_history_turns`), with tags `<ctx>` / `<user>` / `<assistant>`;
- **response** = answer of the assistant to the prompt.

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>", "<user>", "<assistant>", "<ctx>"]
PAD, BOS, EOS, UNK, USR, AST, CTX = SPECIAL_TOKENS


def build_pairs(dataset, max_history_turns=6):
    pairs = []
    for ex in dataset:
        ctx = (ex.get("context") or "").strip()
        turns = ex["conversation"]
        history = []
        for turn in turns:
            role, text = turn["role"], (turn["text"] or "").strip()
            if role == "assistant" and text:
                if history:  # it needs at least one message before the response
                    hist = history[-max_history_turns:]
                    src_parts = []
                    if ctx:
                        src_parts.append(f"{CTX} {ctx}")
                    for h_role, h_text in hist:
                        tag = USR if h_role == "user" else AST
                        src_parts.append(f"{tag} {h_text}")
                    src = " ".join(src_parts).strip()
                    if src:
                        pairs.append((src, text))
            history.append((role, text))
    return pairs


pairs = build_pairs(raw_data, max_history_turns=CONFIG["max_history_turns"])
print("Number of pairs (prompt, response) generated:", len(pairs))
print("\nExample :")
print("PROMPT :", pairs[0][0][:300])
print("RESPONSE:", pairs[0][1][:300])

random.seed(42)
random.shuffle(pairs)
n_val = max(1, int(len(pairs) * CONFIG["val_ratio"]))
val_pairs = pairs[:n_val]
train_pairs = pairs[n_val:]
print(f"\nTrain : {len(train_pairs)} pairs | Val : {len(val_pairs)} pairs")

## 6. BPE Tokenizer trained from scratch

Tokenizer: **byte-level BPE** trained on corpus.

In [ ]:
!pip install -q tokenizers

In [ ]:
import os
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tokenizer_path = os.path.join(CONFIG["checkpoint_dir"], "tokenizer.json")

if os.path.exists(tokenizer_path):
    print("Existing Tokenizer found on Drive, loading...")
    tokenizer = Tokenizer.from_file(tokenizer_path)
else:
    print("Training of a new BPE tokenizer on the corpus...")
    corpus_path = "/content/corpus.txt"
    with open(corpus_path, "w", encoding="utf-8") as f:
        for src, tgt in train_pairs:
            f.write(src.replace("\n", " ") + "\n")
            f.write(tgt.replace("\n", " ") + "\n")

    tokenizer = Tokenizer(models.BPE(unk_token=UNK))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    bpe_trainer = trainers.BpeTrainer(
        vocab_size=CONFIG["vocab_size"],
        special_tokens=SPECIAL_TOKENS,
        min_frequency=2,
    )
    tokenizer.train([corpus_path], bpe_trainer)
    tokenizer.save(tokenizer_path)
    print("Tokenizer saved ->", tokenizer_path)

PAD_ID = tokenizer.token_to_id(PAD)
BOS_ID = tokenizer.token_to_id(BOS)
EOS_ID = tokenizer.token_to_id(EOS)
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE)
print("pad_id:", PAD_ID, "| bos_id:", BOS_ID, "| eos_id:", EOS_ID)
print("\nTokenization example :", tokenizer.encode(pairs[0][1]).tokens)

## 7. PyTorch Dataset — Single Sequence + Prompt Masking

For each example, we construct a single sequence:

`<bos>` + `prompt` + `response` + `<eos>`

The model sees everything (causal self-attention across the entire sequence), but we calculate the **loss only on the tokens in the response** (+ `<eos>`): the prompt tokens are labeled `-100` (PyTorch’s `ignore_index`), so the model is never penalized for “predicting” the prompt.

The prompt is truncated from the left (keeping the end, which is the most recent part) if it exceeds `prompt_max_len`, to ensure that there is always a response left to learn from.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


def encode_example(prompt_text, target_text, tokenizer, prompt_max_len, target_max_len):
    prompt_ids = tokenizer.encode(prompt_text).ids[-prompt_max_len:]   # cut the beginning of the prompt if exceed prompt_max_len
    target_ids = tokenizer.encode(target_text).ids[:target_max_len]
    full_ids = [BOS_ID] + prompt_ids + target_ids + [EOS_ID]
    resp_start = 1 + len(prompt_ids)  # index (in full_ids) of the first token of the response
    return full_ids, resp_start


class ConvDataset(Dataset): # PyTorch Dataset: transforms the text dataset in usable objects for the training process
    def __init__(self, pairs, tokenizer, prompt_max_len, target_max_len):
        self.pairs = pairs
        self.tok = tokenizer
        self.prompt_max_len = prompt_max_len
        self.target_max_len = target_max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return encode_example(src, tgt, self.tok, self.prompt_max_len, self.target_max_len)


def collate_fn(batch, pad_id, ignore_index=-100): # fills the batchs that are shorter with <pad> for the prompt and ignore for the label
    full_list, resp_starts = zip(*batch)
    max_full = max(len(f) for f in full_list)
    seq_len = max_full - 1  # input_ids length = labels (shifted by 1)

    input_batch, label_batch = [], []
    for full_ids, resp_start in zip(full_list, resp_starts):
        inp = full_ids[:-1] # label[t] = full_ids[t+1] only if this token is part of the reponse (or <eos>)
        lab = [full_ids[t + 1] if (t + 1) >= resp_start else ignore_index for t in range(len(full_ids) - 1)]
        pad_len = seq_len - len(inp)
        inp = inp + [pad_id] * pad_len
        lab = lab + [ignore_index] * pad_len
        input_batch.append(inp)
        label_batch.append(lab)
    return torch.tensor(input_batch, dtype=torch.long), torch.tensor(label_batch, dtype=torch.long)


train_ds = ConvDataset(train_pairs, tokenizer, CONFIG["prompt_max_len"], CONFIG["response_max_len"])
val_ds = ConvDataset(val_pairs, tokenizer, CONFIG["prompt_max_len"], CONFIG["response_max_len"])

train_loader = DataLoader(
    train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
    collate_fn=lambda b: collate_fn(b, PAD_ID), drop_last=True, num_workers=2,
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
    collate_fn=lambda b: collate_fn(b, PAD_ID), num_workers=2,
)

inp_b, lab_b = next(iter(train_loader))
print("input_ids:", inp_b.shape, "| labels:", lab_b.shape)
print("\nExample (1st sequence of the batch):")
print("input_ids:", inp_b[0].tolist())
print("labels   :", lab_b[0].tolist())

## 8. Model: Decoder Transformer (RoPE, RMSNorm, SwiGLU, KV cache)

- `precompute_rope_cos_sin` / `apply_rope` implement **RoPE**: the query and key vectors are rotated by an angle that depends on their position—the resulting dot product then depends only on the relative position between two tokens.
- `CausalSelfAttention` handles both training (the entire sequence at once) and generation using a **KV cache** (`kv_cache`): keys and values already computed for previous tokens are reused, and only the new token is computed — generation goes from O(n²) (recalculated at each step) to amortized O(n).

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        norm = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(norm + self.eps)
        return x * self.weight


def precompute_rope_cos_sin(dim, max_len, theta=10000.0):
    """Precompute cos/sin for RoPE."""
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))  # (dim/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)          # (max_len, dim/2)
    emb = torch.cat([freqs, freqs], dim=-1)   # (max_len, dim)
    return emb.cos(), emb.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    # x: (B, H, L, d_k) ; cos/sin: (L, d_k)
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


def make_causal_mask(q_len, k_len, past_len, device):
    # mask[i, j] = True if the query i (absolute position = past_len+i) can see the key j
    i = torch.arange(q_len, device=device).unsqueeze(1)
    j = torch.arange(k_len, device=device).unsqueeze(0)
    allowed = j <= (past_len + i)
    return allowed.unsqueeze(0).unsqueeze(0)  # (1,1,q_len,k_len)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask, cos, sin, kv_cache=None):
        B, L, _ = x.shape
        q = self.w_q(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        k = self.w_k(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        v = self.w_v(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if kv_cache is not None and kv_cache[0] is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v) if kv_cache is not None else None

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(~mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, L, -1)
        return self.w_o(out), new_cache


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)  # gate
        self.w2 = nn.Linear(d_model, d_ff, bias=False)  # value
        self.w3 = nn.Linear(d_ff, d_model, bias=False)  # exit
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class Block(nn.Module):
    """Causal Self-attention (Pre-RMSNorm) + SwiGLU (Pre-RMSNorm)."""

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, mask, cos, sin, kv_cache=None):
        h, new_cache = self.attn(self.norm1(x), mask, cos, sin, kv_cache)
        x = x + h
        x = x + self.ff(self.norm2(x))
        return x, new_cache


class DecoderTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=384, n_heads=6, n_layers=6, d_ff=1024,
                 max_len=1024, dropout=0.1, pad_id=0, rope_theta=10000.0):
        super().__init__()
        self.pad_id = pad_id
        self.d_k = d_model // n_heads
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.embed.weight

        cos, sin = precompute_rope_cos_sin(self.d_k, max_len, rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, ids, kv_cache=None):
        B, L = ids.shape
        device = ids.device
        if kv_cache is not None and kv_cache[0][0] is not None:
            past_len = kv_cache[0][0].shape[2]
        else:
            past_len = 0

        x = self.drop(self.embed(ids))
        cos = self.rope_cos[past_len:past_len + L].to(device)
        sin = self.rope_sin[past_len:past_len + L].to(device)
        mask = make_causal_mask(L, past_len + L, past_len, device)

        new_caches = []
        for i, block in enumerate(self.blocks):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, new_cache = block(x, mask, cos, sin, layer_cache)
            new_caches.append(new_cache)

        x = self.norm_f(x)
        logits = self.head(x)
        return logits, new_caches

    def init_cache(self):
        return [(None, None) for _ in self.blocks]

    @torch.no_grad()
    def generate(self, prompt_ids, bos_id, eos_id, max_new_tokens=100, temperature=0.8, top_k=50):
        """Generate one sequence (no batching) from a tokenized prompt (ids list, no <bos>)."""
        self.eval()
        device = next(self.parameters()).device
        ids = torch.tensor([[bos_id] + list(prompt_ids)], dtype=torch.long, device=device)

        logits, cache = self.forward(ids, kv_cache=self.init_cache())
        next_logits = logits[:, -1, :] / temperature
        generated = []

        for _ in range(max_new_tokens):
            step_logits = next_logits.clone()
            if top_k is not None:
                v, _ = torch.topk(step_logits, top_k)
                step_logits = step_logits.masked_fill(step_logits < v[:, [-1]], float("-inf"))
            probs = torch.softmax(step_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # (1,1)
            token_id = next_token.item()
            if token_id == eos_id:
                break
            generated.append(token_id)

            logits, cache = self.forward(next_token, kv_cache=cache)
            next_logits = logits[:, -1, :] / temperature

        self.train()
        return generated

## 9. Model initiation

In [ ]:
model = DecoderTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=CONFIG["d_model"],
    n_heads=CONFIG["n_heads"],
    n_layers=CONFIG["n_layers"],
    d_ff=CONFIG["d_ff"],
    max_len=CONFIG["pos_max_len"],
    dropout=CONFIG["dropout"],
    pad_id=PAD_ID,
    rope_theta=CONFIG["rope_theta"],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {n_params:,}".replace(",", " "))

## 10. Training

- **AdamW** + schedule with three distinct phases:
    1. **Linear Warmup**: Increases the learning rate from 0 to its peak value over the first `warmup_steps` to stabilize training.
    2. **Inverse Square Root Decay**: Standard decay ($1/\sqrt{\text{step}}$) to keep learning efficiently after the peak.
    3. **Steeper Fine-tuning Decay**: A quadratic decay ($1/\text{step}^2$) for the last `finetune_steps` to force the model to settle into a finer minimum.
- **Mixed precision** (`torch.autocast` + `GradScaler`).
- **Gradient accumulation** to simulate larger batches.
- **Automatic resume** from `CONFIG["checkpoint_dir"]` in the Drive.
- `cross_entropy` loss, with `ignore_index=-100`: thanks to masking, the loss only applies to the response tokens.

In [ ]:
import math, time
import torch.nn as nn

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)

d_model_cfg, warmup = CONFIG["d_model"], CONFIG["warmup_steps"]

def lr_lambda(step):
    step = max(step, 1)

    # Define a steeper decay exponent for fine-tuning (2.0 for quadratic decay) for the last 10K steps
    finetune_start_step = CONFIG["max_steps"] - CONFIG["finetune_steps"]
    new_decay_exponent = 2.0

    if step < warmup: # Linear warmup phase
        decay_term = step * warmup ** -1.5
    elif step < finetune_start_step: # Original decay phase (inverse square root)
        decay_term = step ** -0.5
    else: # Steeper decay phase for fine-tuning
        value_at_finetune_start = finetune_start_step ** -0.5 # Ensure continuity at the finetune_start_step
        C = value_at_finetune_start * (finetune_start_step ** new_decay_exponent)
        decay_term = C * (step ** -new_decay_exponent)

    raw = (d_model_cfg ** -0.5) * decay_term
    peak = (d_model_cfg ** -0.5) * (warmup ** -0.5)

    return raw / peak  # normalized for CONFIG["lr"] at peak

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

ckpt_path = os.path.join(CONFIG["checkpoint_dir"], "model_last.pt")
start_step = 0

if os.path.exists(ckpt_path):
    print("Checkpoint found, resuming training...")
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_step = ckpt["step"]
    print(f"Resuming at step {start_step}")
else:
    print("No checkpoint found, training from the start.")


def save_checkpoint(step):
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "step": step,
        "config": CONFIG,
    }, ckpt_path)


@torch.no_grad()
def evaluate(n_batches=20):
    model.eval()
    total_loss, n = 0.0, 0
    for i, (inp, lab) in enumerate(val_loader):
        if i >= n_batches:
            break
        inp, lab = inp.to(device), lab.to(device)
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits, _ = model(inp, kv_cache=None)
            loss = nn.functional.cross_entropy(logits.reshape(-1, VOCAB_SIZE), lab.reshape(-1), ignore_index=-100)
        total_loss += loss.item()
        n += 1
    model.train()
    return total_loss / max(n, 1)

In [ ]:
model.train()
step = start_step
t0 = time.time()
train_iter = iter(train_loader)

while step < CONFIG["max_steps"]:
    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0

    for _ in range(CONFIG["grad_accum_steps"]):
        try:
            inp, lab = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            inp, lab = next(train_iter)

        inp, lab = inp.to(device), lab.to(device)

        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits, _ = model(inp, kv_cache=None)
            loss = nn.functional.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE), lab.reshape(-1), ignore_index=-100
            ) / CONFIG["grad_accum_steps"]

        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
    step += 1

    if step % CONFIG["log_every"] == 0:
        elapsed = time.time() - t0
        print(f"step {step}/{CONFIG['max_steps']} | loss {accum_loss:.4f} | "
              f"lr {scheduler.get_last_lr()[0]:.2e} | {elapsed:.1f}s")
        t0 = time.time()

    if step % CONFIG["save_every"] == 0:
        val_loss = evaluate()
        print(f"  -> val_loss {val_loss:.4f} | val_ppl {math.exp(val_loss):.2f}")
        save_checkpoint(step)
        print(f"  -> checkpoint saved at step {step}")

save_checkpoint(step)
print("Training ended, final checkpoint saved.")

## 11. Text generation / interactive chat (with KV cache)

The prompt is formatted exactly as it was during training, then `model.generate(...)` performs a full pass over the prompt (filling the cache), and then generates one token at a time, recalculating only that new token at each step (KV cache).

**Also usable with only a CPU**

In [ ]:
def format_history(history, max_history_turns=CONFIG["max_history_turns"], ctx=""):
    hist = history[-max_history_turns:]
    parts = []
    if ctx:
        parts.append(f"{CTX} {ctx}")
    for role, text in hist:
        tag = USR if role == "user" else AST
        parts.append(f"{tag} {text}")
    return " ".join(parts).strip()


@torch.no_grad()
def generate_reply(history, ctx="", max_new_tokens=100, temperature=0.8, top_k=50):
    prompt = format_history(history, ctx=ctx)
    prompt_ids = tokenizer.encode(prompt).ids[-CONFIG["prompt_max_len"]:]
    gen_ids = model.generate(prompt_ids, bos_id=BOS_ID, eos_id=EOS_ID,
                              max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


# Example
demo_history = [("user", "Quelle est la capitale de la France ?")]
print("Assistant :", generate_reply(demo_history))

In [ ]:
# Interactive chat (write "exit" to stop)
history = []
print("Discussion avec le modèle (tape 'exit' pour quitter)\n")

while True:
    user_msg = input("Toi : ")
    if user_msg.strip().lower() in ("exit", "quit"):
        break
    history.append(("user", user_msg))
    reply = generate_reply(history)
    print("Assistant :", reply)
    history.append(("assistant", reply))